# Interpolation Operations

This notebook demonstrates how to perform interpolation operations on topological geometries
using topologic_fast.

**Adapted from topologicpy Interpolation example.**

We will:
1. Create a triangulated shell (grid mesh)
2. Define influencing vertices with values
3. Interpolate values across the shell vertices using Inverse Distance Weighting (IDW)
4. Visualize the interpolated values as a color map
5. Create a terrain by displacing vertices based on interpolated values

**NOTE:** topologic_fast does not have built-in `Vertex.InterpolateValue()` or Dictionary support.
We implement interpolation manually using Python.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

print("Libraries imported successfully.")

## 1. Input Parameters

In [ ]:
# Shell parameters
size = 40       # The size of the shell (40 x 40 units)
sides = 20      # The number of divisions in each direction

# Interpolation parameters
key = "intensity"   # The name of the value being interpolated
n_neighbors = 4     # Number of nearest neighbors to consider for interpolation
power = 2           # Power parameter for IDW (higher = more local influence)

print(f"Shell size: {size} x {size}")
print(f"Grid divisions: {sides} x {sides}")
print(f"Interpolation neighbors: {n_neighbors}")

## 2. Create the Shell (Grid Mesh)

We create a rectangular shell divided into triangular faces.
Since topologic_fast may not have `Shell.Rectangle()` with subdivision,
we build the mesh manually.

In [ ]:
def create_rectangular_grid(width, length, u_divisions, v_divisions):
    """
    Create a rectangular grid of triangulated faces.
    
    Returns:
        shell: The Shell topology
        vertices: List of all vertices (row-major order)
    """
    # Calculate spacing
    dx = width / u_divisions
    dy = length / v_divisions
    
    # Create vertices grid
    vertices = []
    for j in range(v_divisions + 1):
        for i in range(u_divisions + 1):
            x = -width/2 + i * dx
            y = -length/2 + j * dy
            z = 0
            v = tf.Vertex.ByCoordinates(x, y, z)
            vertices.append(v)
    
    # Create triangular faces
    faces = []
    for j in range(v_divisions):
        for i in range(u_divisions):
            # Get vertex indices for this cell
            idx00 = j * (u_divisions + 1) + i
            idx10 = j * (u_divisions + 1) + i + 1
            idx01 = (j + 1) * (u_divisions + 1) + i
            idx11 = (j + 1) * (u_divisions + 1) + i + 1
            
            # Get vertices
            v00 = vertices[idx00]
            v10 = vertices[idx10]
            v01 = vertices[idx01]
            v11 = vertices[idx11]
            
            # Create two triangles for each cell
            # Triangle 1: v00, v10, v11
            e1 = tf.Edge.ByVertices(v00, v10)
            e2 = tf.Edge.ByVertices(v10, v11)
            e3 = tf.Edge.ByVertices(v11, v00)
            w1 = tf.Wire.ByEdges([e1, e2, e3])
            f1 = tf.Face.ByWire(w1)
            faces.append(f1)
            
            # Triangle 2: v00, v11, v01
            e4 = tf.Edge.ByVertices(v00, v11)
            e5 = tf.Edge.ByVertices(v11, v01)
            e6 = tf.Edge.ByVertices(v01, v00)
            w2 = tf.Wire.ByEdges([e4, e5, e6])
            f2 = tf.Face.ByWire(w2)
            faces.append(f2)
    
    # Create shell from faces
    shell = tf.Shell.ByFaces(faces)
    
    return shell, vertices

# Create the shell
shell, grid_vertices = create_rectangular_grid(size, size, sides, sides)

print(f"Created Shell:")
print(f"  Grid vertices: {len(grid_vertices)}")
print(f"  Shell faces: {shell.NumFaces()}")
print(f"  Shell area: {shell.Area():.2f} sq units")

## 3. Visualize the Base Shell

In [ ]:
def visualize_shell(shell, title="Shell", color='lightblue'):
    """Create 3D visualization of a shell."""
    fig = go.Figure()
    
    faces = shell.Faces()
    
    for face in faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [c[2] for c in coords]
        
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=0.7,
            alphahull=0,
            showlegend=False,
            hoverinfo='skip'
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        width=700,
        height=600
    )
    
    return fig

fig_shell = visualize_shell(shell, "Base Triangulated Shell")
fig_shell.show()

## 4. Create Influencing Vertices

We define vertices with associated values that will influence the interpolation.
In topologicpy, these would use Dictionary to store the values.
Here we store them as Python data.

In [ ]:
# NOTE: In topologicpy, you would use:
#   v1 = Vertex.ByCoordinates(-10, -10, 0)
#   d1 = Dictionary.ByKeysValues([key], [5])
#   v1 = Topology.SetDictionary(v1, d1)
#
# Since topologic_fast doesn't have Dictionary, we use Python data structures:

class InfluencerVertex:
    """A vertex with an associated influence value."""
    def __init__(self, x, y, z, value):
        self.vertex = tf.Vertex.ByCoordinates(x, y, z)
        self.coords = (x, y, z)
        self.value = value

# Create influencer vertices at corners with alternating values
influencers = [
    InfluencerVertex(-10, -10, 0, 5),   # Corner 1: positive
    InfluencerVertex(10, -10, 0, -5),   # Corner 2: negative
    InfluencerVertex(10, 10, 0, 5),     # Corner 3: positive
    InfluencerVertex(-10, 10, 0, -5),   # Corner 4: negative
]

print("Influencing Vertices:")
print("-" * 50)
for i, inf in enumerate(influencers):
    print(f"  V{i+1}: position={inf.coords}, {key}={inf.value}")

## 5. Implement Inverse Distance Weighting (IDW) Interpolation

IDW interpolates values based on the weighted average of nearby points,
where weights are inversely proportional to distance.

Formula: `value = sum(w_i * v_i) / sum(w_i)`
where `w_i = 1 / d_i^p` (d = distance, p = power parameter)

In [ ]:
# NOTE: In topologicpy, you would use:
#   s_v = Vertex.InterpolateValue(s_v, influencers, n, key=key, tolerance=0.0001)
#   d = Topology.Dictionary(s_v)
#   value = Dictionary.ValueAtKey(d, key)
#
# We implement IDW interpolation manually:

def distance(p1, p2):
    """Calculate Euclidean distance between two points."""
    return np.sqrt(sum((a - b)**2 for a, b in zip(p1, p2)))

def interpolate_idw(target_coords, influencers, n_neighbors=4, power=2, tolerance=0.0001):
    """
    Interpolate value at target point using Inverse Distance Weighting.
    
    Parameters:
        target_coords: (x, y, z) of target point
        influencers: List of InfluencerVertex objects
        n_neighbors: Number of nearest neighbors to consider
        power: Power parameter (higher = more local influence)
        tolerance: Distance below which we return exact value
    
    Returns:
        Interpolated value
    """
    # Calculate distances to all influencers
    distances_values = []
    for inf in influencers:
        d = distance(target_coords, inf.coords)
        distances_values.append((d, inf.value))
    
    # Sort by distance and take nearest n_neighbors
    distances_values.sort(key=lambda x: x[0])
    nearest = distances_values[:n_neighbors]
    
    # Check if we're very close to an influencer
    if nearest[0][0] < tolerance:
        return nearest[0][1]
    
    # Calculate weighted average
    weighted_sum = 0
    weight_sum = 0
    
    for d, v in nearest:
        weight = 1.0 / (d ** power)
        weighted_sum += weight * v
        weight_sum += weight
    
    return weighted_sum / weight_sum

# Interpolate values for all grid vertices
interpolated_values = []

for vertex in grid_vertices:
    coords = vertex.Coordinates()
    value = interpolate_idw(coords, influencers, n_neighbors, power)
    interpolated_values.append(value)

print(f"Interpolated {len(interpolated_values)} vertex values")
print(f"Value range: [{min(interpolated_values):.3f}, {max(interpolated_values):.3f}]")

## 6. Visualize Interpolated Values (Color Map)

In [ ]:
def visualize_interpolation_2d(grid_vertices, values, influencers, u_div, v_div, title="Interpolated Values"):
    """Create 2D heatmap visualization of interpolated values."""
    
    # Extract coordinates
    coords = [v.Coordinates() for v in grid_vertices]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    # Reshape for heatmap
    z_grid = np.array(values).reshape(v_div + 1, u_div + 1)
    
    # Create x and y arrays for heatmap
    x_unique = sorted(list(set(x)))
    y_unique = sorted(list(set(y)))
    
    fig = go.Figure()
    
    # Add heatmap
    fig.add_trace(go.Heatmap(
        x=x_unique,
        y=y_unique,
        z=z_grid,
        colorscale='RdYlBu_r',  # Red-Yellow-Blue reversed (thermal-like)
        colorbar=dict(title=key),
        hovertemplate='X: %{x:.1f}<br>Y: %{y:.1f}<br>Value: %{z:.3f}<extra></extra>'
    ))
    
    # Add influencer points
    inf_x = [inf.coords[0] for inf in influencers]
    inf_y = [inf.coords[1] for inf in influencers]
    inf_vals = [inf.value for inf in influencers]
    
    fig.add_trace(go.Scatter(
        x=inf_x,
        y=inf_y,
        mode='markers+text',
        marker=dict(size=20, color='white', line=dict(color='black', width=3)),
        text=[f"{v}" for v in inf_vals],
        textposition='middle center',
        textfont=dict(size=12, color='black'),
        name='Influencers',
        hoverinfo='text',
        hovertext=[f"Influencer: {key}={v}" for v in inf_vals]
    ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=700,
        height=600
    )
    
    return fig

fig_2d = visualize_interpolation_2d(grid_vertices, interpolated_values, influencers, sides, sides, "IDW Interpolation")
fig_2d.show()

## 7. Create Terrain (Displaced Mesh)

We use the interpolated values to displace vertices in the Z direction,
creating a terrain surface.

In [ ]:
# NOTE: In topologicpy, you would use:
#   new_vertices = []
#   for s_v in shell_vertices:
#       d = Topology.Dictionary(s_v)
#       value = Dictionary.ValueAtKey(d, key)
#       new_v = Vertex.ByCoordinates(Vertex.X(s_v), Vertex.Y(s_v), value)
#       new_vertices.append(new_v)
#   new_shell = Topology.ReplaceVertices(shell, shell_vertices, new_vertices)
#
# We create a new shell with displaced vertices:

def create_terrain_shell(grid_vertices, values, u_div, v_div, z_scale=1.0):
    """
    Create a terrain shell by displacing vertices based on values.
    
    Parameters:
        grid_vertices: Original grid vertices
        values: Interpolated values for each vertex
        u_div, v_div: Grid divisions
        z_scale: Scale factor for Z displacement
    
    Returns:
        shell: New Shell with displaced vertices
        new_vertices: List of displaced vertices
    """
    # Create displaced vertices
    new_vertices = []
    for v, val in zip(grid_vertices, values):
        coords = v.Coordinates()
        new_v = tf.Vertex.ByCoordinates(coords[0], coords[1], val * z_scale)
        new_vertices.append(new_v)
    
    # Create triangular faces using the new vertices
    faces = []
    for j in range(v_div):
        for i in range(u_div):
            # Get vertex indices
            idx00 = j * (u_div + 1) + i
            idx10 = j * (u_div + 1) + i + 1
            idx01 = (j + 1) * (u_div + 1) + i
            idx11 = (j + 1) * (u_div + 1) + i + 1
            
            v00 = new_vertices[idx00]
            v10 = new_vertices[idx10]
            v01 = new_vertices[idx01]
            v11 = new_vertices[idx11]
            
            # Triangle 1
            e1 = tf.Edge.ByVertices(v00, v10)
            e2 = tf.Edge.ByVertices(v10, v11)
            e3 = tf.Edge.ByVertices(v11, v00)
            w1 = tf.Wire.ByEdges([e1, e2, e3])
            f1 = tf.Face.ByWire(w1)
            faces.append(f1)
            
            # Triangle 2
            e4 = tf.Edge.ByVertices(v00, v11)
            e5 = tf.Edge.ByVertices(v11, v01)
            e6 = tf.Edge.ByVertices(v01, v00)
            w2 = tf.Wire.ByEdges([e4, e5, e6])
            f2 = tf.Face.ByWire(w2)
            faces.append(f2)
    
    shell = tf.Shell.ByFaces(faces)
    return shell, new_vertices

# Create terrain
terrain_shell, terrain_vertices = create_terrain_shell(grid_vertices, interpolated_values, sides, sides)

print(f"Created terrain shell:")
print(f"  Vertices: {len(terrain_vertices)}")
print(f"  Faces: {terrain_shell.NumFaces()}")

## 8. Visualize Terrain

In [ ]:
def visualize_terrain(terrain_vertices, values, u_div, v_div, title="Interpolated Terrain"):
    """Create 3D surface visualization of terrain."""
    
    # Extract coordinates
    coords = [v.Coordinates() for v in terrain_vertices]
    
    # Reshape for surface plot
    x_grid = np.array([c[0] for c in coords]).reshape(v_div + 1, u_div + 1)
    y_grid = np.array([c[1] for c in coords]).reshape(v_div + 1, u_div + 1)
    z_grid = np.array([c[2] for c in coords]).reshape(v_div + 1, u_div + 1)
    
    fig = go.Figure()
    
    fig.add_trace(go.Surface(
        x=x_grid,
        y=y_grid,
        z=z_grid,
        colorscale='RdYlBu_r',
        colorbar=dict(title=key),
        contours=dict(
            z=dict(show=True, usecolormap=True, highlightcolor='white', project_z=True)
        )
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.3),
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title=key,
            camera=dict(eye=dict(x=1.5, y=-1.5, z=0.8))
        ),
        width=800,
        height=700
    )
    
    return fig

fig_terrain = visualize_terrain(terrain_vertices, interpolated_values, sides, sides)
fig_terrain.show()

## 9. Side-by-Side Comparison

In [ ]:
# Create side-by-side visualization
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=['Flat Shell', 'Interpolated Terrain']
)

# Extract coordinates for flat and terrain
flat_coords = [v.Coordinates() for v in grid_vertices]
terrain_coords = [v.Coordinates() for v in terrain_vertices]

# Reshape
flat_x = np.array([c[0] for c in flat_coords]).reshape(sides + 1, sides + 1)
flat_y = np.array([c[1] for c in flat_coords]).reshape(sides + 1, sides + 1)
flat_z = np.zeros((sides + 1, sides + 1))

terrain_x = np.array([c[0] for c in terrain_coords]).reshape(sides + 1, sides + 1)
terrain_y = np.array([c[1] for c in terrain_coords]).reshape(sides + 1, sides + 1)
terrain_z = np.array([c[2] for c in terrain_coords]).reshape(sides + 1, sides + 1)

# Color by interpolated values for both
value_grid = np.array(interpolated_values).reshape(sides + 1, sides + 1)

# Add flat surface (colored by interpolated values)
fig.add_trace(
    go.Surface(
        x=flat_x, y=flat_y, z=flat_z,
        surfacecolor=value_grid,
        colorscale='RdYlBu_r',
        showscale=False
    ),
    row=1, col=1
)

# Add terrain surface
fig.add_trace(
    go.Surface(
        x=terrain_x, y=terrain_y, z=terrain_z,
        colorscale='RdYlBu_r',
        showscale=True,
        colorbar=dict(title=key, x=1.02)
    ),
    row=1, col=2
)

fig.update_layout(
    title='Interpolation: Flat vs Terrain',
    width=1200,
    height=600
)

# Update scene cameras
camera = dict(eye=dict(x=1.2, y=-1.2, z=0.8))
fig.update_scenes(camera=camera)

fig.show()

## 10. Experiment: More Influencers

In [ ]:
# Add more influencers for a more complex interpolation
influencers_extended = [
    InfluencerVertex(-10, -10, 0, 5),
    InfluencerVertex(10, -10, 0, -5),
    InfluencerVertex(10, 10, 0, 5),
    InfluencerVertex(-10, 10, 0, -5),
    # Center point
    InfluencerVertex(0, 0, 0, 8),
    # Edge midpoints
    InfluencerVertex(0, -15, 0, -3),
    InfluencerVertex(0, 15, 0, -3),
    InfluencerVertex(-15, 0, 0, 2),
    InfluencerVertex(15, 0, 0, 2),
]

# Interpolate with extended influencers
values_extended = []
for vertex in grid_vertices:
    coords = vertex.Coordinates()
    value = interpolate_idw(coords, influencers_extended, n_neighbors=5, power=2)
    values_extended.append(value)

# Create terrain
terrain_ext, verts_ext = create_terrain_shell(grid_vertices, values_extended, sides, sides)

# Visualize
fig_ext = visualize_terrain(verts_ext, values_extended, sides, sides, "Extended Influencers Terrain")
fig_ext.show()

## Summary

This notebook demonstrated interpolation operations using topologic_fast.

### Key Operations:

1. **Grid Mesh Creation**: Built a triangulated shell using `tf.Face.ByWire()` and `tf.Shell.ByFaces()`
2. **IDW Interpolation**: Implemented Inverse Distance Weighting manually in Python
3. **Terrain Generation**: Created displaced mesh by modifying Z coordinates
4. **Visualization**: Used Plotly for 2D heatmaps and 3D surfaces

### Differences from topologicpy:

- **No `Shell.Rectangle()` with subdivision**: Built grid manually
- **No `Topology.Triangulate()`**: Created triangles explicitly
- **No `Vertex.InterpolateValue()`**: Implemented IDW interpolation manually
- **No Dictionary support**: Stored values in Python data structures
- **No `Topology.ReplaceVertices()`**: Rebuilt topology with new vertices
- **No `Topology.Show()` with colorScale**: Used Plotly directly

### Applications:

- Terrain modeling and analysis
- Spatial data interpolation
- Heat/gradient visualization
- Environmental simulation
- Geographic Information Systems (GIS)